# MCLDNN Differential Attention Training and Comparison

This notebook trains the new `mcldnn_diffattention` model and compares it against the normal `mcldnn_attention` model.

Main rule followed here:

- Differential-attention results are exported/downloaded.
- Normal-attention results are used only for comparison plots/tables and are not included in the final zip.

The comparison uses:

- test accuracy
- test accuracy vs SNR
- training accuracy vs SNR
- mean accuracy over all SNRs
- peak SNR accuracy
- parameter count

Reuse behavior:

- If `experiments/5class_diffattention/checkpoints/best_model.weights.h5` and the required result CSVs already exist, the notebook skips differential-attention training.
- If an old downloaded result folder such as `diffattention_results_YYYYMMDD_HHMM` is attached/pasted, the notebook copies it into `experiments/5class_diffattention` first.
- The final zip is repo-ready: extracting it at the repo root recreates `experiments/5class_diffattention/...`.


In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink

os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/akshlabh/amr-5-class.git"
WORK_DIR = Path("/kaggle/working/amr-5-class")
DATASET = Path("/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat")

# If internet is available, clone/pull. If not, use an already attached repo copy if present.
def find_attached_repo():
    for root in Path("/kaggle/input").glob("**"):
        if (root / "src" / "train.py").exists() and (root / "configs").exists():
            return root
    return None

if (WORK_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(WORK_DIR), "pull"], check=False)
elif WORK_DIR.exists() and (WORK_DIR / "src" / "train.py").exists():
    print(f"Using existing work dir: {WORK_DIR}")
else:
    attached = find_attached_repo()
    if attached is not None:
        print(f"Copying attached repo from: {attached}")
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print("Cloning repo from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print(f"Working dir: {Path.cwd()}")
print(f"Dataset    : {DATASET}")
print(f"Dataset OK : {DATASET.exists()}")



DIFF_DIR = Path("experiments/5class_diffattention")
NORMAL_DIR = Path("experiments/5class_attention")
MODEL_VARIANT = "gated_diffattention_v3_qam_weighted"
VARIANT_FILE = Path("results/model_variant.txt")


def has_current_variant_marker(path: Path) -> bool:
    marker = path / VARIANT_FILE
    return marker.exists() and marker.read_text().strip() == MODEL_VARIANT


def looks_like_diff_results(path: Path) -> bool:
    """Reuse only results from the current tuned diff-attention variant."""
    return (
        (path / "checkpoints" / "best_model.weights.h5").exists()
        and (path / "results" / "test_score.csv").exists()
        and (path / "results" / "acc_per_snr.csv").exists()
        and has_current_variant_marker(path)
    )


def find_existing_diff_results():
    """Find already-produced current diff-attention results in common Kaggle locations."""
    candidates = []
    candidates.append(DIFF_DIR)
    for base in [Path("/kaggle/working"), Path("/kaggle/input"), Path("experiments")]:
        if base.exists():
            candidates.extend(base.glob("**/diffattention_results*"))
            candidates.extend(base.glob("**/5class_diffattention"))

    seen = set()
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.add(cand)
        if looks_like_diff_results(cand):
            return cand
    return None


def copy_existing_diff_results_to_repo(src: Path, dst: Path = DIFF_DIR):
    """Copy existing current-variant result folder into canonical repo location."""
    src = src.resolve()
    dst = dst.resolve()
    if src == dst:
        print(f"Using existing diff-attention results at: {dst}")
        return
    print(f"Copying existing diff-attention results from {src} to {dst}")
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


required = [
    "src/models/mcldnn_attention.py",
    "src/models/mcldnn_diffattention.py",
    "configs/exp_5class_attention.yaml",
    "configs/exp_5class_diffattention.yaml",
    "src/train.py",
]
for f in required:
    print(f"{'?' if Path(f).exists() else '? MISSING'} {f}")

In [ ]:
# CELL 2: Quick model shape and parameter check
import gc
import keras
import keras.backend as K

keras.mixed_precision.set_global_policy("float32")
K.clear_session(); gc.collect()

from src.models.mcldnn_attention import build_mcldnn_attention
from src.models.mcldnn_diffattention import (
    build_mcldnn_diffattention,
    build_mcldnn_diffattention_extractor,
)

normal_model = build_mcldnn_attention(classes=5)
diff_model = build_mcldnn_diffattention(classes=5)
diff_extractor = build_mcldnn_diffattention_extractor(classes=5)

print(f"Normal attention params : {normal_model.count_params():,}")
print(f"Diff attention params   : {diff_model.count_params():,}")
print(f"Under 300k              : {'YES' if diff_model.count_params() < 300_000 else 'NO'}")

x1 = np.zeros((4, 2, 128, 1), dtype="float32")
x2 = np.zeros((4, 128, 1), dtype="float32")
x3 = np.zeros((4, 128, 1), dtype="float32")

pred = diff_model.predict([x1, x2, x3], verbose=0)
pred2, attn = diff_extractor.predict([x1, x2, x3], verbose=0)

print(f"Diff training output shape  : {pred.shape}")
print(f"Diff extractor softmax shape: {pred2.shape}")
print(f"Diff attention map shape    : {attn.shape}")
print(f"Diff attention min/max      : {attn.min():.5f}, {attn.max():.5f}")

assert pred.shape == (4, 5)
assert pred2.shape == (4, 5)
assert attn.shape == (4, 2, 124, 124)
assert diff_model.count_params() < 300_000

print("Shape checks passed")

del normal_model, diff_model, diff_extractor
K.clear_session(); gc.collect()

In [ ]:
# CELL 3: Train Differential Attention model from scratch with live epoch logs
# Always retrains. Existing diff-attention outputs are removed first so old weights are not reused.

import os, sys, subprocess, shutil
from pathlib import Path

assert DATASET.exists(), f"Dataset not found: {DATASET}"

DIFF_WEIGHTS = DIFF_DIR / "checkpoints" / "best_model.weights.h5"
DIFF_TEST_SCORE = DIFF_DIR / "results" / "test_score.csv"
DIFF_SNR = DIFF_DIR / "results" / "acc_per_snr.csv"

if DIFF_DIR.exists():
    print(f"Removing existing diff-attention outputs before fresh training: {DIFF_DIR}")
    shutil.rmtree(DIFF_DIR)

print("Training differential attention from scratch...\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    [
        sys.executable, "-u", "src/train.py",
        "--config", "configs/exp_5class_diffattention.yaml",
        "--datasetpath", str(DATASET),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

(DIFF_DIR / "results").mkdir(parents=True, exist_ok=True)
(DIFF_DIR / VARIANT_FILE).write_text(MODEL_VARIANT)

print("\nDifferential attention training finished.")
print("Checkpoint:", DIFF_WEIGHTS.exists())
print("Test score:", DIFF_TEST_SCORE.exists())
print("SNR CSV   :", DIFF_SNR.exists())
print("Variant   :", (DIFF_DIR / VARIANT_FILE).read_text().strip() if (DIFF_DIR / VARIANT_FILE).exists() else "missing")

In [ ]:
# CELL 4: Ensure normal attention results are available for comparison
# This cell trains normal attention only if its result CSV is missing.
# Normal-attention outputs are NOT included in the final download zip.

NORMAL_RESULT = NORMAL_DIR / "results" / "test_score.csv"
NORMAL_SNR = NORMAL_DIR / "results" / "acc_per_snr.csv"

if NORMAL_RESULT.exists() and NORMAL_SNR.exists():
    print("Normal attention results already exist. Using them for comparison only.")
else:
    print("Normal attention results not found. Training normal attention for comparison only...")
    subprocess.run([
        "python", "src/train.py",
        "--config", "configs/exp_5class_attention.yaml",
        "--datasetpath", str(DATASET),
    ], check=True)

print("Normal attention test score exists:", NORMAL_RESULT.exists())
print("Normal attention SNR CSV exists  :", NORMAL_SNR.exists())

In [ ]:
# CELL 5: Compare Normal Attention vs Differential Attention
COMPARE_DIR = DIFF_DIR / "comparison_with_normal_attention"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

normal_score = pd.read_csv(NORMAL_DIR / "results" / "test_score.csv")
diff_score = pd.read_csv(DIFF_DIR / "results" / "test_score.csv")

normal_snr = pd.read_csv(NORMAL_DIR / "results" / "acc_per_snr.csv")
diff_snr = pd.read_csv(DIFF_DIR / "results" / "acc_per_snr.csv")

normal_snr = normal_snr.rename(columns={"accuracy": "normal_attention_acc"})
diff_snr = diff_snr.rename(columns={"accuracy": "diff_attention_acc"})
comparison = pd.merge(normal_snr, diff_snr, on="snr", how="inner")
comparison["delta_diff_minus_normal"] = comparison["diff_attention_acc"] - comparison["normal_attention_acc"]
comparison["normal_attention_acc_percent"] = 100.0 * comparison["normal_attention_acc"]
comparison["diff_attention_acc_percent"] = 100.0 * comparison["diff_attention_acc"]
comparison["delta_percent_points"] = 100.0 * comparison["delta_diff_minus_normal"]

summary = pd.DataFrame([
    {
        "model": "normal_attention",
        "test_loss": float(normal_score.loc[0, "loss"]),
        "test_accuracy": float(normal_score.loc[0, "accuracy"]),
        "mean_snr_accuracy": float(comparison["normal_attention_acc"].mean()),
        "peak_snr_accuracy": float(comparison["normal_attention_acc"].max()),
        "peak_snr_db": int(comparison.loc[comparison["normal_attention_acc"].idxmax(), "snr"]),
    },
    {
        "model": "diff_attention",
        "test_loss": float(diff_score.loc[0, "loss"]),
        "test_accuracy": float(diff_score.loc[0, "accuracy"]),
        "mean_snr_accuracy": float(comparison["diff_attention_acc"].mean()),
        "peak_snr_accuracy": float(comparison["diff_attention_acc"].max()),
        "peak_snr_db": int(comparison.loc[comparison["diff_attention_acc"].idxmax(), "snr"]),
    },
])

comparison_csv = COMPARE_DIR / "normal_vs_diffattention_acc_per_snr.csv"
summary_csv = COMPARE_DIR / "normal_vs_diffattention_summary.csv"
comparison.to_csv(comparison_csv, index=False)
summary.to_csv(summary_csv, index=False)

print("Summary:")
display(summary)
print("\nPer-SNR comparison:")
display(comparison)
print(f"Saved: {comparison_csv}")
print(f"Saved: {summary_csv}")

In [ ]:
# CELL 6: Plot comparison curves
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(comparison["snr"], comparison["normal_attention_acc_percent"],
        marker="o", linewidth=2.4, label="Normal attention")
ax.plot(comparison["snr"], comparison["diff_attention_acc_percent"],
        marker="s", linewidth=2.4, label="Differential attention")
ax.set_title("Normal Attention vs Differential Attention: Accuracy vs SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(comparison["snr"])
plt.tight_layout()
curve_path = COMPARE_DIR / "normal_vs_diffattention_acc_vs_snr.png"
fig.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {curve_path}")

fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in comparison["delta_percent_points"]]
ax.bar(comparison["snr"].astype(str), comparison["delta_percent_points"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Differential Attention minus Normal Attention by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delta accuracy (percentage points)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
delta_path = COMPARE_DIR / "diff_minus_normal_delta_by_snr.png"
fig.savefig(delta_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {delta_path}")

In [ ]:
# CELL 7: Compare training accuracy vs SNR curves
# This evaluates the saved best weights on the training split, grouped by SNR.
# It does not download normal-attention outputs; it only saves comparison files under diff-attention results.

import keras.backend as K
from src.dataset import load_data, FIVE_CLASS
from src.models.mcldnn_attention import build_mcldnn_attention
from src.models.mcldnn_diffattention import build_mcldnn_diffattention

K.clear_session(); gc.collect()

(mods_train, snrs_train, lbl_train), (X_train, Y_train), _, _, (train_idx, _, _) = load_data(
    str(DATASET), FIVE_CLASS, seed=2016, shuffle_split=False
)
train_SNRs = np.array([lbl_train[i][1] for i in train_idx])
TRAIN_SNRS = sorted(set(train_SNRs))
print(f"Training set: {X_train.shape[0]} samples across {len(TRAIN_SNRS)} SNRs")


def make_inputs(X):
    return [
        np.expand_dims(X, axis=3).astype("float32"),
        np.expand_dims(X[:, 0, :], axis=2).astype("float32"),
        np.expand_dims(X[:, 1, :], axis=2).astype("float32"),
    ]


def eval_by_snr(model, inputs, Y, snr_labels, snr_values, batch_size=400):
    accs = {}
    for snr in snr_values:
        mask = snr_labels == snr
        X_snr = [branch[mask] for branch in inputs]
        Y_snr = Y[mask]
        score = model.evaluate(X_snr, Y_snr, verbose=0, batch_size=batch_size)
        accs[snr] = float(score[1])
    return accs

inp_train = make_inputs(X_train)
normal_weights = NORMAL_DIR / "checkpoints" / "best_model.weights.h5"
diff_weights = DIFF_DIR / "checkpoints" / "best_model.weights.h5"
assert normal_weights.exists(), f"Missing normal attention weights: {normal_weights}"
assert diff_weights.exists(), f"Missing diff attention weights: {diff_weights}"

normal_train_model = build_mcldnn_attention(classes=len(mods_train))
normal_train_model.load_weights(normal_weights)
normal_train_accs = eval_by_snr(normal_train_model, inp_train, Y_train, train_SNRs, TRAIN_SNRS)
print("Normal attention train mean acc:", np.mean(list(normal_train_accs.values())))

del normal_train_model
K.clear_session(); gc.collect()

diff_train_model = build_mcldnn_diffattention(classes=len(mods_train))
diff_train_model.load_weights(diff_weights)
diff_train_accs = eval_by_snr(diff_train_model, inp_train, Y_train, train_SNRs, TRAIN_SNRS)
print("Diff attention train mean acc:", np.mean(list(diff_train_accs.values())))

del diff_train_model
K.clear_session(); gc.collect()

train_comparison = pd.DataFrame({
    "snr": TRAIN_SNRS,
    "normal_attention_train_acc": [normal_train_accs[s] for s in TRAIN_SNRS],
    "diff_attention_train_acc": [diff_train_accs[s] for s in TRAIN_SNRS],
})
train_comparison["delta_diff_minus_normal"] = train_comparison["diff_attention_train_acc"] - train_comparison["normal_attention_train_acc"]
train_comparison["normal_attention_train_acc_percent"] = 100.0 * train_comparison["normal_attention_train_acc"]
train_comparison["diff_attention_train_acc_percent"] = 100.0 * train_comparison["diff_attention_train_acc"]
train_comparison["delta_percent_points"] = 100.0 * train_comparison["delta_diff_minus_normal"]

train_comparison_csv = COMPARE_DIR / "normal_vs_diffattention_train_acc_per_snr.csv"
train_comparison.to_csv(train_comparison_csv, index=False)
print(f"Saved: {train_comparison_csv}")
display(train_comparison)

# Plot training accuracy vs SNR
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(train_comparison["snr"], train_comparison["normal_attention_train_acc_percent"],
        marker="o", linewidth=2.4, label="Normal attention train")
ax.plot(train_comparison["snr"], train_comparison["diff_attention_train_acc_percent"],
        marker="s", linewidth=2.4, label="Differential attention train")
ax.set_title("Training Accuracy vs SNR: Normal Attention vs Differential Attention")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Training Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(train_comparison["snr"])
plt.tight_layout()
train_curve_path = COMPARE_DIR / "normal_vs_diffattention_train_acc_vs_snr.png"
fig.savefig(train_curve_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {train_curve_path}")

# Plot training delta by SNR
fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in train_comparison["delta_percent_points"]]
ax.bar(train_comparison["snr"].astype(str), train_comparison["delta_percent_points"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Training Delta: Differential Attention minus Normal Attention by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delta training accuracy (percentage points)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
train_delta_path = COMPARE_DIR / "diff_minus_normal_train_delta_by_snr.png"
fig.savefig(train_delta_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {train_delta_path}")

# Plot train vs test curves for both models
train_test = pd.merge(
    train_comparison[["snr", "normal_attention_train_acc_percent", "diff_attention_train_acc_percent"]],
    comparison[["snr", "normal_attention_acc_percent", "diff_attention_acc_percent"]],
    on="snr",
    how="inner",
)
train_test_csv = COMPARE_DIR / "normal_vs_diffattention_train_test_acc_per_snr.csv"
train_test.to_csv(train_test_csv, index=False)

fig, ax = plt.subplots(figsize=(12, 6.5))
ax.plot(train_test["snr"], train_test["normal_attention_train_acc_percent"],
        marker="o", linewidth=2.2, linestyle="-", label="Normal train")
ax.plot(train_test["snr"], train_test["normal_attention_acc_percent"],
        marker="o", linewidth=2.2, linestyle="--", label="Normal test")
ax.plot(train_test["snr"], train_test["diff_attention_train_acc_percent"],
        marker="s", linewidth=2.2, linestyle="-", label="Diff train")
ax.plot(train_test["snr"], train_test["diff_attention_acc_percent"],
        marker="s", linewidth=2.2, linestyle="--", label="Diff test")
ax.set_title("Training vs Test Accuracy by SNR\nSolid = train, dashed = test")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(train_test["snr"])
plt.tight_layout()
train_vs_test_path = COMPARE_DIR / "normal_vs_diffattention_train_vs_test_acc_by_snr.png"
fig.savefig(train_vs_test_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {train_test_csv}")
print(f"Saved: {train_vs_test_path}")

In [ ]:
# CELL 8: Copy comparison files into diff-attention results folder
# They are saved under the diff-attention experiment so the final zip is self-contained.
# Normal attention checkpoints/results are not copied.

DIFF_COMPARE_EXPORT = DIFF_DIR / "results" / "comparison_with_normal_attention"
DIFF_COMPARE_EXPORT.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    comparison_csv,
    summary_csv,
    curve_path,
    delta_path,
    train_comparison_csv,
    train_test_csv,
    train_curve_path,
    train_delta_path,
    train_vs_test_path,
]

for src in files_to_copy:
    dst = DIFF_COMPARE_EXPORT / Path(src).name
    shutil.copy2(src, dst)
    print(f"Copied: {dst}")

In [ ]:
# CELL 9: Create repo-ready zip for ONLY differential-attention results
# The zip contains: experiments/5class_diffattention/...
# Extract it at the repo root and paths will line up consistently with other notebooks.

stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = Path("/kaggle/working") / f"diffattention_repo_ready_{stamp}"
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(WORK_DIR),
    base_dir="experiments/5class_diffattention",
)

print(f"Created repo-ready zip: {zip_path}")
print("\nExtract this zip at the repo root. It will create/update:")
print("  experiments/5class_diffattention/")
print("\nIncluded files:")
for path in sorted(DIFF_DIR.rglob("*")):
    if path.is_file():
        print(" -", Path("experiments/5class_diffattention") / path.relative_to(DIFF_DIR))

display(FileLink(zip_path))